<a href="https://colab.research.google.com/github/nermal1/Stock-Market-Prediction-437/blob/main/EnsembleReal.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Import Libraries

In [33]:
import yfinance as yf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.compose import ColumnTransformer
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

from sklearn.ensemble import StackingClassifier
from sklearn.ensemble import RandomForestClassifier, VotingClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.svm import SVC
from sklearn.naive_bayes import GaussianNB

from sklearn.linear_model import LogisticRegression

## Indicator Functions

In [13]:
def weighted_moving_average(data, period):
  weights = np.arange(1, period + 1)
  return data.rolling(period).apply(lambda x: np.dot(x, weights) / weights.sum(), raw=True)

In [14]:
def featureSelection(df):

  df = df.copy()

  # Basic Returns
  df['Return'] = df['Close'].pct_change()

  # Technical Indicators
  df['SMA_14'] = df['Close'].rolling(window=14).mean()
  df['SMA_50'] = df['Close'].rolling(window=50).mean()
  df['WMA_14'] = weighted_moving_average(df['Close'], 14)
  df['Momentum_10'] = df['Close'] / df['Close'].shift(10) - 1
  df['Volatility_14'] = df['Return'].rolling(window=14).std()

  # RSI Calculation
  delta = df['Close'].diff()
  gain = (delta.where(delta > 0, 0))
  loss = (-delta.where(delta < 0, 0))
  avg_gain = gain.rolling(window=14).mean()
  avg_loss = loss.rolling(window=14).mean()
  rs = avg_gain / avg_loss
  df['RSI_14'] = 100 - (100 / (1 + rs))

  # Lags (Previous days' returns)
  lags = [1, 2, 3, 5]
  for lag in lags:
    df[f'Lag_{lag}'] = df['Return'].shift(lag)

  # Target: 1 if Up, 0 if Down
  df['Target'] = np.where(df['Return'] > 0, 1, 0)

  return df.dropna()

## Ticker Configurations
Give each ticker its own model

In [26]:
TICKER_CONFIGS = {
    'AAPL': {
        'weights': [3, 4, 0, 1],
        'nb': {'features': ['SMA_50', 'WMA_14', 'Momentum_10', 'Lag_3', 'Lag_5']},
        'svm': {'features': ['Momentum_10', 'Lag_1', 'Lag_2'], 'C': 0.5},
        'rf': {'features': ['RSI_14', 'Momentum_10', 'Lag_5'], 'n_estimators': 100},
        'ann': {'features': ['Momentum_10', 'Lag_1', 'Lag_2'], 'hidden_layer_sizes': (100,), 'momentum': 0.6}
    },
    'MSFT': {
        'weights': [2, 1, 1, 1],
        'nb': {'features': ['RSI_14', 'Momentum_10', 'Lag_5']},
        'svm': {'features': ['RSI_14', 'Momentum_10', 'Lag_1', 'Lag_2', 'Lag_5'], 'C': 0.5},
        'rf': {'features': ['RSI_14', 'Volatility_14', 'Lag_2','Lag_5'], 'n_estimators': 50},
        'ann': {'features': ['Momentum_10', 'Volatility_14', 'Lag_2', 'Lag_5'], 'hidden_layer_sizes': (10,), 'momentum': 0.9}
    },
    '^GSPC': {
        'weights': [3, 3, 0, 1],
        'nb': {'features': ['Momentum_10', 'Volatility_14', 'Lag_1', 'Lag_3', 'Lag_5']},
        'svm': {'features': ['Momentum_10', 'Lag_1', 'Lag_2', 'Lag_5'], 'C': 5},
        'rf': {'features': ['SMA_50', 'Momentum_10', 'Volatility_14'], 'n_estimators': 150},
        'ann': {'features': ['Momentum_10', 'Lag_1', 'Lag_2', 'Lag_5'], 'hidden_layer_sizes': (100,), 'momentum': 0.9}
    },
    '^DJI': {
        'weights': [2, 4, 0, 1],
        'nb': {'features': ['Momentum_10', 'Volatility_14', 'Lag_1', 'Lag_2', 'Lag_3', 'Lag_5']},
        'svm': {'features': ['Momentum_10', 'Volatility_14', 'Lag_1', 'Lag_2', 'Lag_5'], 'C': 1},
        'rf': {'features': ['RSI_14', 'Momentum_10', 'Lag_5'], 'n_estimators': 100},
        'ann': {'features': ['Momentum_10', 'Lag_1', 'Lag_2', 'Lag_5'], 'hidden_layer_sizes': (10,), 'momentum': 0.3}
    }
}

## Fetch Data

In [27]:
tickers = list(TICKER_CONFIGS.keys())
data_store = {}

print("Fetching Data...")
for ticker in tickers:
    raw_df = yf.download(ticker, start="2010-01-01", end="2019-12-31", progress=False, auto_adjust=True)
    if isinstance(raw_df.columns, pd.MultiIndex):
        raw_df = raw_df.xs(ticker, axis=1, level=1)
    data_store[ticker] = featureSelection(raw_df)

Fetching Data...


In [28]:
def get_selector(features):
    return ColumnTransformer(
        [('selector', 'passthrough', features)],
        remainder='drop'
    )

In [36]:
print("\nStarting Stacking Ensemble Training")

for ticker, df in data_store.items():
  print(f"\nProcessing {ticker}...")

  config = TICKER_CONFIGS.get(ticker)

  # Get all unique features needed across all models
  all_needed_features = set()
  for model_name in ['nb', 'svm', 'rf', 'ann']:
    all_needed_features.update(config[model_name]['features'])

  feature_cols = list(all_needed_features)

  X = df[feature_cols]
  y = df['Target']
  X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, shuffle=False)

  # Create pipelines for each base model with their specific features
  nb_pipe = make_pipeline(
    get_selector(config['nb']['features']),
    GaussianNB()
  )

  svm_pipe = make_pipeline(
    get_selector(config['svm']['features']),
    StandardScaler(),
    SVC(C=config['svm']['C'], gamma='scale', probability=True, random_state=42)
  )

  rf_pipe = make_pipeline(
    get_selector(config['rf']['features']),
    RandomForestClassifier(n_estimators=config['rf'].get('n_estimators', 100), random_state=42)
    )

  ann_pipe = make_pipeline(
    get_selector(config['ann']['features']),
    StandardScaler(),
    MLPClassifier(
      hidden_layer_sizes=config['ann'].get('hidden_layer_sizes', (50,)),
      activation='relu',
      solver='sgd',
      learning_rate_init=0.1,
      momentum=config['ann'].get('momentum', 0.9),
      max_iter=config['ann'].get('max_iter', 2000),
      random_state=42,
      early_stopping=True
      )
  )

  # Evaluate individual models
  models = {'NB': nb_pipe, 'SVM': svm_pipe, 'RF': rf_pipe, 'ANN': ann_pipe}
  best_single_score = 0

  print("Individual Performance:")
  for name, model in models.items():
    model.fit(X_train, y_train)
    score = model.score(X_test, y_test)
    print(f"  {name}: {score:.4f}")
    if score > best_single_score:
      best_single_score = score

    # Try multiple meta-learners
  meta_learners = {
    'Logistic Regression': LogisticRegression(random_state=42, max_iter=1000),
    'Random Forest': RandomForestClassifier(n_estimators=50, max_depth=5, random_state=42),
    'Gradient Boosting': GaussianNB()
  }

  print("\nStacking Ensemble Results:")
  best_stacking_score = 0
  best_stacking_pred = None
  best_meta_name = None

  for meta_name, meta_learner in meta_learners.items():
    stacking = StackingClassifier(
      estimators=[
        ('nb', nb_pipe),
        ('svm', svm_pipe),
        ('rf', rf_pipe),
        ('ann', ann_pipe)
        ],
        final_estimator=meta_learner,
        cv=5,  # Use cross validation to generate meta-features
        n_jobs=-1,
        passthrough=False
    )

    stacking.fit(X_train, y_train)
    y_pred = stacking.predict(X_test)
    acc = accuracy_score(y_test, y_pred)

    print(f"\n  Meta-learner: {meta_name}")
    print(f"  Accuracy: {acc:.2%}")

    if acc > best_single_score:
      print(f"  Result: SUCCESS (Beats best single model by {acc - best_single_score:.2%})")
    else:
      print(f"  Result: LOWER (Best single model is higher by {best_single_score - acc:.2%})")

    if acc > best_stacking_score:
      best_stacking_score = acc
      best_stacking_pred = y_pred
      best_meta_name = meta_name

  # Print detailed metrics for best stacking ensemble
  if best_stacking_pred is not None:
    prec = precision_score(y_test, best_stacking_pred, zero_division=0)
    rec = recall_score(y_test, best_stacking_pred, zero_division=0)
    f1 = f1_score(y_test, best_stacking_pred, zero_division=0)

    print(f"\n  Best Stacking Model: {best_meta_name}")
    print(f"  Best Accuracy:     {best_stacking_score:.2%}")
    print(f"  Precision:         {prec:.4f}")
    print(f"  Recall:            {rec:.4f}")
    print(f"  F1 Score:          {f1:.4f}")


Starting Stacking Ensemble Training

Processing AAPL...
Individual Performance:
  NB: 0.6397
  SVM: 0.6478
  RF: 0.5870
  ANN: 0.6579

Stacking Ensemble Results:

  Meta-learner: Logistic Regression
  Accuracy: 65.59%
  Result: LOWER (Best single model is higher by 0.20%)

  Meta-learner: Random Forest
  Accuracy: 64.78%
  Result: LOWER (Best single model is higher by 1.01%)

  Meta-learner: Gradient Boosting
  Accuracy: 63.97%
  Result: LOWER (Best single model is higher by 1.82%)

  Best Stacking Model: Logistic Regression
  Best Accuracy:     65.59%
  Precision:         0.6554
  Recall:            0.7860
  F1 Score:          0.7148

Processing MSFT...
Individual Performance:
  NB: 0.6275
  SVM: 0.6377
  RF: 0.6215
  ANN: 0.6518

Stacking Ensemble Results:

  Meta-learner: Logistic Regression
  Accuracy: 65.79%
  Result: SUCCESS (Beats best single model by 0.61%)

  Meta-learner: Random Forest
  Accuracy: 67.41%
  Result: SUCCESS (Beats best single model by 2.23%)

  Meta-learner: G